# LS-LMSR Mechanism Demo
**Preclinical Prediction Markets — MOLECULA**

This notebook demonstrates the core pricing mechanism: the Liquidity-Sensitive Logarithmic Market Scoring Rule (LS-LMSR) with Automated Bioactivity Market Maker (ABMM) cold-start seeding.

Live platform: [molecula-flame.vercel.app](https://molecula-flame.vercel.app)  
GitHub: [adityanbhosale/lmsr-preclinical-markets](https://github.com/adityanbhosale/lmsr-preclinical-markets)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0d1117'
plt.rcParams['axes.facecolor'] = '#161b22'
plt.rcParams['axes.edgecolor'] = '#30363d'
plt.rcParams['axes.labelcolor'] = '#c9d1d9'
plt.rcParams['xtick.color'] = '#8b949e'
plt.rcParams['ytick.color'] = '#8b949e'
plt.rcParams['text.color'] = '#c9d1d9'
plt.rcParams['grid.color'] = '#21262d'
plt.rcParams['grid.linewidth'] = 0.8
plt.rcParams['font.family'] = 'monospace'

## 1. LS-LMSR Core Math

The cost function:
$$C(\mathbf{q}) = b(\mathbf{q}) \cdot \log\left(\sum_i \exp\left(\frac{q_i}{b(\mathbf{q})}\right)\right)$$

Where liquidity scales with volume:
$$b(\mathbf{q}) = \alpha \cdot \sum_i q_i$$

Marginal price (probability estimate):
$$p_i(\mathbf{q}) = \frac{\partial C}{\partial q_i} = \frac{\exp(q_i / b)}{\sum_j \exp(q_j / b)}$$

Note: when $\mathbf{q} = (0,0)$, $b = 0$ and the cost function is undefined. Markets initialize at $\mathbf{q} = (0.01, 0.01)$ to keep $b$ well-defined before ABMM seeding.

Alpha is oracle-derived per molecule:
$$\alpha = 0.005 + (1 - \text{confidence\_score}) \times 0.075$$

In [ ]:
def lslmsr_price(q_yes, q_no, alpha):
    """LS-LMSR marginal price for YES outcome."""
    b = alpha * (q_yes + q_no)
    exp_yes = np.exp(q_yes / b)
    exp_no  = np.exp(q_no  / b)
    return exp_yes / (exp_yes + exp_no)

def lslmsr_cost(q_yes, q_no, alpha):
    """LS-LMSR cost function."""
    b = alpha * (q_yes + q_no)
    return b * np.log(np.exp(q_yes / b) + np.exp(q_no / b))

# Three molecules with different confidence scores
molecules = {
    'GNF-4471 (high confidence, α=0.012)':   {'alpha': 0.012, 'color': '#58a6ff'},
    'ARV-771 (mid confidence, α=0.040)':     {'alpha': 0.040, 'color': '#3fb950'},
    'PROTACore-12 (low confidence, α=0.075)': {'alpha': 0.075, 'color': '#f78166'},
}

print("Alpha derivation: alpha = 0.005 + (1 - confidence_score) * 0.075")
print(f"  High confidence (0.90): alpha = {0.005 + (1-0.90)*0.075:.3f}")
print(f"  Mid  confidence (0.55): alpha = {0.005 + (1-0.55)*0.075:.3f}")
print(f"  Low  confidence (0.00): alpha = {0.005 + (1-0.00)*0.075:.3f}")

## 2. Price Curves: Effect of Alpha on Market Sensitivity

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle('LS-LMSR Price Curves by Confidence Score', fontsize=14, y=1.02)

q_no_fixed = 1.0
q_yes_range = np.linspace(0.01, 5.0, 300)

for ax, (name, props) in zip(axes, molecules.items()):
    prices = [lslmsr_price(q, q_no_fixed, props['alpha']) for q in q_yes_range]
    ax.plot(q_yes_range, prices, color=props['color'], linewidth=2)
    ax.axhline(0.5, color='#8b949e', linestyle='--', linewidth=0.8, label='p=0.5')
    ax.axvline(q_no_fixed, color='#8b949e', linestyle=':', linewidth=0.8, label='q_no')
    ax.set_title(name, fontsize=9, pad=10)
    ax.set_xlabel('q_yes (shares staked)')
    ax.set_ylabel('P(YES)' if ax == axes[0] else '')
    ax.set_ylim(0, 1)
    ax.grid(True)
    ax.legend(fontsize=8)

plt.tight_layout()
plt.show()
print("Higher alpha = flatter curve = less sensitive to trades = more stable price for uncertain molecules")

## 3. ABMM Cold-Start Seeding & Retreat

The ABMM holds initial YES/NO stakes seeded from historical PoS priors (Hay et al., 2014).  
As credentialed human volume accumulates, the LDI (Liquidity Depth Index) drives ABMM retreat:

$$w(t) = e^{-\lambda \cdot \text{LDI}_{\text{calibrated}}(t)}$$

where $\lambda = \log(2) / \text{ldi\_half}$ sets the half-life of ABMM influence.

In [ ]:
def abmm_influence(ldi, decay_rate=3.0):
    """Exponential retreat: ABMM influence as function of LDI."""
    return np.exp(-decay_rate * ldi)

def abmm_influence_linear(ldi):
    return np.maximum(0, 1 - ldi)

def abmm_influence_convex(ldi, power=0.5):
    return np.maximum(0, 1 - ldi**power)

ldi_range = np.linspace(0, 1, 300)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ABMM Retreat Functions', fontsize=13)

# Left: retreat function comparison
ax1.plot(ldi_range, abmm_influence(ldi_range),        color='#58a6ff', lw=2.5, label='Exponential (preferred)')
ax1.plot(ldi_range, abmm_influence_linear(ldi_range), color='#8b949e', lw=1.5, linestyle='--', label='Linear')
ax1.plot(ldi_range, abmm_influence_convex(ldi_range), color='#f78166', lw=1.5, linestyle=':', label='Convex')
ax1.set_xlabel('LDI (Liquidity Depth Index)')
ax1.set_ylabel('ABMM Influence')
ax1.set_title('Retreat Function Comparison')
ax1.legend()
ax1.grid(True)
ax1.set_ylim(0, 1.05)

# Right: simulated price path as experts trade
np.random.seed(42)
n_trades = 40
alpha = 0.040  # ARV-771

# ABMM seeds at IND→Approval overall prior: ~10.4% from Hay et al.
prior_yes = 0.104
q_yes = prior_yes * 2.0
q_no  = (1 - prior_yes) * 2.0
ldi = 0.0

prices     = [lslmsr_price(q_yes, q_no, alpha)]
ldis       = [ldi]
influences = [1.0]

for i in range(n_trades):
    trade = np.random.normal(0.08, 0.15)
    if trade > 0:
        q_yes += trade
    else:
        q_no  += abs(trade)
    ldi = min(1.0, ldi + np.random.uniform(0.015, 0.035))
    prices.append(lslmsr_price(q_yes, q_no, alpha))
    ldis.append(ldi)
    influences.append(abmm_influence(ldi))

trade_idx = range(len(prices))
ax2_twin  = ax2.twinx()

ax2.plot(trade_idx, prices, color='#3fb950', lw=2, label='P(IND) — ARV-771')
ax2.axhline(prior_yes, color='#8b949e', lw=1, linestyle='--', label=f'Hay et al. prior ({prior_yes:.1%})')
ax2_twin.plot(trade_idx, influences, color='#f78166', lw=1.5, linestyle=':', label='ABMM influence')
ax2_twin.set_ylabel('ABMM Influence', color='#f78166')
ax2_twin.tick_params(axis='y', labelcolor='#f78166')
ax2_twin.set_ylim(0, 1.2)

ax2.set_xlabel('Credentialed Trade #')
ax2.set_ylabel('Implied P(IND Submission)')
ax2.set_title('Live Price Path: Expert Trading Displaces ABMM')
ax2.set_ylim(0, 0.6)
ax2.grid(True)

lines1, labels1 = ax2.get_legend_handles_labels()
lines2, labels2 = ax2_twin.get_legend_handles_labels()
ax2.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='upper left')

plt.tight_layout()
plt.show()

## 4. Historical Prior Calibration (Hay et al., 2014)

ABMM opening prices are seeded from phase transition rates. Note the distinction:
- **IND → Approval (overall)**: 10.4% — the cumulative probability of full approval from IND submission, used as the M1 anchor
- **Phase 1→2, 2→3, 3→Approval**: conditional transition rates between adjacent stages, used to seed M2–M4 markets

These are not the same quantity and should not be compared directly on the same scale.

In [ ]:
# Hay et al. 2014 — phase transition rates
# M1 anchor: overall IND→Approval probability (cumulative)
# M2-M4 anchors: conditional phase transition rates

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('ABMM Prior Calibration — Hay et al. (2014)', fontsize=13)

# Left: M1 anchor — overall cumulative approval probability
indications = ['All indications', 'Oncology', 'Hematology', 'Infectious disease']
overall_pos  = [0.104, 0.057, 0.116, 0.191]
colors = ['#58a6ff', '#f78166', '#3fb950', '#d2a8ff']
bars = ax1.bar(indications, overall_pos, color=colors, alpha=0.85)
for bar, val in zip(bars, overall_pos):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
             f'{val:.1%}', ha='center', va='bottom', fontsize=10)
ax1.set_ylabel('P(Approval | IND submission)')
ax1.set_title('M1 Anchor: Overall IND → Approval\n(cumulative probability, used to seed M1 market)')
ax1.set_ylim(0, 0.28)
ax1.grid(True, axis='y')

# Right: M2-M4 anchors — conditional phase transition rates (all indications)
transitions = ['Ph1 → Ph2', 'Ph2 → Ph3', 'Ph3 → Approval']
transition_rates_all  = [0.595, 0.368, 0.582]
transition_rates_onco = [0.544, 0.285, 0.503]
x = np.arange(len(transitions))
width = 0.35
b1 = ax2.bar(x - width/2, transition_rates_all,  width, label='All indications', color='#58a6ff', alpha=0.85)
b2 = ax2.bar(x + width/2, transition_rates_onco, width, label='Oncology',        color='#f78166', alpha=0.85)
for bar in list(b1) + list(b2):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
             f'{bar.get_height():.1%}', ha='center', va='bottom', fontsize=9)
ax2.set_xticks(x)
ax2.set_xticklabels(transitions)
ax2.set_ylabel('Conditional Phase Transition Probability')
ax2.set_title('M2–M4 Anchors: Conditional Phase Transitions\n(used to seed M2, M3, M4 markets respectively)')
ax2.legend()
ax2.set_ylim(0, 0.80)
ax2.grid(True, axis='y')

plt.tight_layout()
plt.show()
print("M1 uses cumulative IND→Approval probability as its opening price anchor.")
print("M2–M4 use conditional transition rates — these are stage-specific, not cumulative.")

## 5. Open Problem: ABMM Incentive Compatibility

The exponential retreat schedule is heuristically motivated but its incentive-compatibility properties under the Transaction Fee Mechanism (TFM) framework of Roughgarden et al. (2023) remain an open question.

Specifically: does the ABMM's sequential position reduction satisfy myopic best-response incentive compatibility (MBIC) for credentialed expert traders? The challenge is that ABMM retreat is a function of aggregate credentialed volume — creating a potential coordination incentive where expert traders time entries to exploit predictable ABMM withdrawal.

The formal condition for ε-incentive-compatibility requires:

$$w(t) \cdot q_{\text{abmm}} \leq \delta(\varepsilon, \alpha, p^*)$$

where $\delta$ is tightest when the expert's true belief $p^*$ is far from the ABMM's seeded price.

This is an open problem. Contributions welcome — see `docs/mechanism.md` for a formal statement.